# Simple Medicine Recommendation System

### Project idea
Recommend medicines whose **Reason + Description** are most similar to a user's text query.

This is a **text similarity / information retrieval project**, not a medical diagnosis or prescribing system.

### NLP used
Only basic NLP:
- lowercase text
- stemming/lemmatization
- TF-IDF
- cosine similarity

In [ ]:
import pandas as pd
import re
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def clean_and_stem(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    words = text.split()
    words = [stemmer.stem(word) for word in words]
    return " ".join(words)


In [ ]:
# Load dataset
data = pd.read_csv("Medicine_description(Sheet1).csv", encoding="latin1")

df = data.copy()

print("Shape:", df.shape)
print(df.columns.tolist())
df.head()


Shape: (22481, 3)
['Drug_Name', 'Reason', 'Description']


,Drug_Name,Reason,Description
0,A CN Gel(Topical) 20gmA CN Soap 75gm,Acne,Mild to moderate acne (spots)
1,A Ret 0.05% Gel 20gmA Ret 0.1% Gel 20gmA Ret 0...,Acne,A RET 0.025% is a prescription medicine that i...
2,ACGEL CL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
3,ACGEL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
4,Acleen 1% Lotion 25ml,Acne,treat the most severe form of acne (nodular ac...


In [ ]:
# Check missing values
print(df.isnull().sum())

# Fill missing text values
df["Drug_Name"] = df["Drug_Name"].fillna("")
df["Reason"] = df["Reason"].fillna("")
df["Description"] = df["Description"].fillna("")


Drug_Name       0
Reason          0
Description    14
dtype: int64


## 1. Basic text cleaning

We only convert text to lowercase and remove unnecessary symbols.

No advanced NLP is needed for this project.


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

# Basic cleaning + stemming
df["Reason_clean"] = df["Reason"].apply(clean_and_stem)
df["Description_clean"] = df["Description"].apply(clean_and_stem)

df[["Reason", "Reason_clean"]].head()


,Reason,Reason_clean
0,Acne,acn
1,Acne,acn
2,Acne,acn
3,Acne,acn
4,Acne,acn


## 2. Create one text column

We combine **Reason** and **Description** because both contain information about the medicine.

The recommendation will be based on textual similarity between the user's query and these medicine records.


In [ ]:
df["text"] = df["Reason_clean"] + " " + df["Description_clean"]

df[["Drug_Name", "Reason", "Description", "text"]].head()


,Drug_Name,Reason,Description,text
0,A CN Gel(Topical) 20gmA CN Soap 75gm,Acne,Mild to moderate acne (spots),acn mild to moder acn spot
1,A Ret 0.05% Gel 20gmA Ret 0.1% Gel 20gmA Ret 0...,Acne,A RET 0.025% is a prescription medicine that i...,acn a ret 0 025 is a prescript medicin that is...
2,ACGEL CL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...,acn it is use to treat acn vulgari in peopl 12...
3,ACGEL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...,acn it is use to treat acn vulgari in peopl 12...
4,Acleen 1% Lotion 25ml,Acne,treat the most severe form of acne (nodular ac...,acn treat the most sever form of acn nodular acn


## 3. Convert text into TF-IDF vectors

TF-IDF gives more importance to words that are useful for distinguishing one medicine record from another.

We use only **single words (unigrams)** to keep the project simple.


In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")

tfidf_matrix = vectorizer.fit_transform(df["text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF matrix shape: (22481, 735)


## 4. Recommendation function

When a user enters a sentence:

1. Clean the query.
2. Convert it to a TF-IDF vector using the **already fitted vectorizer**.
3. Calculate cosine similarity with every medicine.
4. Return the medicines with the highest similarity.


In [ ]:
def recommend_medicines(query, top_n=5):
    query = clean_text(query)

    query_vector = vectorizer.transform([query])

    similarity = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = np.argsort(similarity)[::-1][:top_n]

    result = df.iloc[top_indices][
        ["Drug_Name", "Reason", "Description"]
    ].copy()

    result["Similarity"] = similarity[top_indices]

    return result.reset_index(drop=True)


## 5. Test the recommender

Try different natural-language queries.


In [ ]:
recommend_medicines("I have acne with pimples and blackheads")


,Drug_Name,Reason,Description,Similarity
0,A CN Gel(Topical) 20gmA CN Soap 75gm,Acne,Mild to moderate acne (spots),0.0
1,Acnelak Z Lotion 15gm,Acne,treat the most severe form of acne (nodular ac...,0.0
2,Acnemoist Cream 60gm,Acne,treat the most severe form of acne (nodular ac...,0.0
3,Acnerex Soap 75gm,Acne,treat acne vulgaris,0.0
4,Acneril 1% Gel 10gmAcneril Tablet 10Acneril 0....,Acne,treat acne vulgaris,0.0


In [ ]:
recommend_medicines("I have high blood pressure")


,Drug_Name,Reason,Description,Similarity
0,Angibloc AM 50/5mg Tablet 10'S,Hypertension,lower high blood pressure (hypertension),0.470959
1,Esloril 2.5/2.5mg Tablet 10'S,Hypertension,lower high blood pressure (hypertension),0.470959
2,Telfon 40mg Tablet 10'S,Hypertension,lower high blood pressure (hypertension),0.470959
3,Telfirst 20mg Tablet 10'STelfirst 80mg Tablet ...,Hypertension,lower high blood pressure (hypertension),0.470959
4,Espin 2.5mg Tablet 10'SEspin 5mg Tablet 10'S,Hypertension,lower high blood pressure (hypertension),0.470959


In [ ]:
recommend_medicines("I have a fungal infection on my skin")


,Drug_Name,Reason,Description,Similarity
0,Unigat Gel 15gm,Fungal,recommended for fungal infections,0.519019
1,Terbiclair Cream 10gm,Fungal,recommended for fungal infections,0.519019
2,Zypower Tablet 4's,Fungal,recommended for fungal infections,0.519019
3,Voxpert 200mg Tablet 4'S,Fungal,recommended for fungal infections,0.519019
4,Trucand DT 50mg Tablet 4'S,Fungal,recommended for fungal infections,0.519019


In [ ]:
recommend_medicines("I have wounds on my skin")


,Drug_Name,Reason,Description,Similarity
0,Delpoclav DS 457mg Dry Syrup 30ml,Infection,used for skin and skin structure,0.722077
1,Dalacin C 150mg Capsule 10'SDalacin C 600mg In...,Infection,used for skin and skin structure,0.722077
2,Cumin CV Tablet 6'SCumin CV Tablet 10'S,Infection,used for skin and skin structure,0.722077
3,Cozan O Tablet 10'S,Infection,used for skin and skin structure,0.722077
4,CONIDERM 200mg Capsule 4's,Infection,used for skin and skin structure,0.722077


## 6. Understand the result

The **Similarity** score is based on cosine similarity.

- Higher similarity → the medicine text is more similar to the user's query.
- Lower similarity → less textual similarity.

This does **not** mean the medicine is medically appropriate for the user.


In [ ]:
# See the top 10 results for one query
recommend_medicines("wounds on skin", top_n=10)


,Drug_Name,Reason,Description,Similarity
0,Delpoclav DS 457mg Dry Syrup 30ml,Infection,used for skin and skin structure,0.722077
1,Dalacin C 150mg Capsule 10'SDalacin C 600mg In...,Infection,used for skin and skin structure,0.722077
2,Cumin CV Tablet 6'SCumin CV Tablet 10'S,Infection,used for skin and skin structure,0.722077
3,Cozan O Tablet 10'S,Infection,used for skin and skin structure,0.722077
4,CONIDERM 200mg Capsule 4's,Infection,used for skin and skin structure,0.722077
5,Coligyl Dry Syrup 30ml,Infection,used for skin and skin structure,0.722077
6,Cloxaid Capsule 10'SCloxaid Injection 5ml,Infection,used for skin and skin structure,0.722077
7,Clobetamil Cream 10gm,Infection,used for skin and skin structure,0.722077
8,Clid 300mg Injection 2mlClid 300mg Injection 4...,Infection,used for skin and skin structure,0.722077
9,Clavmentin Injection 1'SClavmentin 625mg Table...,Infection,used for skin and skin structure,0.722077


## 7. Simple manual evaluation

We can test the system using a few queries for which we know the expected **Reason/category**.

This is a basic project evaluation rather than a sophisticated NLP evaluation.


In [ ]:
test_queries = [
    "acne pimples",
    "high blood pressure",
    "fungal infection",
    "skin wounds"
]

for query in test_queries:
    print("\nQuery:", query)
    result = recommend_medicines(query, top_n=3)
    print(result[["Drug_Name", "Reason", "Similarity"]])



Query: acne pimples
                              Drug_Name Reason  Similarity
0  A CN Gel(Topical) 20gmA CN Soap 75gm   Acne         0.0
1                 Acnelak Z Lotion 15gm   Acne         0.0
2                  Acnemoist Cream 60gm   Acne         0.0

Query: high blood pressure
                        Drug_Name        Reason  Similarity
0  Angibloc AM 50/5mg Tablet 10'S  Hypertension    0.470959
1   Esloril 2.5/2.5mg Tablet 10'S  Hypertension    0.470959
2         Telfon 40mg Tablet 10'S  Hypertension    0.470959

Query: fungal infection
                                           Drug_Name  Reason  Similarity
0  Seriva Cream 30gmSeriva Cream 10gmSeriva 2% Lo...  Fungal    0.703736
1             Sertaset Cream 30gmSertaset Cream 15gm  Fungal    0.703736
2                     SPEXIDIME Dusting Powder 100gm  Fungal    0.703736

Query: skin wounds
                                           Drug_Name     Reason  Similarity
0                  Delpoclav DS 457mg Dry Syrup 30ml  Infectio

## 9. Optional: Transformer-based similarity

TF-IDF mainly looks at individual words. A transformer model can create a numerical representation (embedding) that captures more of the meaning of a sentence.

For this project, we use a **pre-trained Sentence Transformer**. We do not train the transformer ourselves.

This keeps the transformer part simple: **text → embedding → cosine similarity**.

> This section is optional. For the interview, focus first on TF-IDF, stemming and cosine similarity.

In [ ]:
# Install once if needed:
# pip install sentence-transformers

from sentence_transformers import SentenceTransformer

transformer_model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
# Create embeddings for the medicine text
medicine_embeddings = transformer_model.encode(
    df["text"].tolist(),
    show_progress_bar=True
)


In [ ]:
def recommend_with_transformer(query, top_n=5):
    query_embedding = transformer_model.encode([query])

    similarity = cosine_similarity(
        query_embedding,
        medicine_embeddings
    ).flatten()

    top_indices = np.argsort(similarity)[::-1][:top_n]

    result = df.iloc[top_indices][
        ["Drug_Name", "Reason", "Description"]
    ].copy()

    result["Similarity"] = similarity[top_indices]
    return result.reset_index(drop=True)

recommend_with_transformer("I have acne and pimples")




**Basic version:**
Query → cleaning → stemming → TF-IDF → cosine similarity → recommendations

**Transformer version:**
Query → pre-trained transformer → sentence embedding → cosine similarity → recommendations
